In [ ]:
# Cell 1
!pip install qrcode[pil] Pillow numpy

# Cell 2 — update your URL
URL = "https://huggingface.co/spaces/kittu125/FamilyBasedOilBlend"

In [ ]:
import torch
import qrcode
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
import os

# -----------------------------
# Settings
# -----------------------------

MODEL_CACHE = "./models"
OUTPUT_DIR = "./output"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROMPT = """
premium cooking oil advertisement,
olive oil bottle with olives and avocado,
green natural background,
qr code hidden in artwork
"""

NEGATIVE_PROMPT = "blurry, distorted qr, broken pattern"

QR_CONTENT = "https://huggingface.co/spaces/kittu125/FamilyBasedOilBlend"

NUM_IMAGES = 4

GUIDANCE = 7
CONTROL_SCALE = 1.8
STEPS = 40

BOX_SIZE = 16
BORDER = 6

# -----------------------------
# Load Model (only once)
# -----------------------------

print("Loading ControlNet...")

controlnet = ControlNetModel.from_pretrained(
    "monster-labs/control_v1p_sd15_qrcode_monster",
    cache_dir=MODEL_CACHE,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    cache_dir=MODEL_CACHE,
    safety_checker=None,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

if DEVICE == "cuda":
    pipe.enable_xformers_memory_efficient_attention()

# -----------------------------
# Generate QR Base
# -----------------------------

qr = qrcode.QRCode(
    version=1,
    error_correction=qrcode.constants.ERROR_CORRECT_H,
    box_size=BOX_SIZE,
    border=BORDER
)

qr.add_data(QR_CONTENT)
qr.make(fit=True)

qr_img = qr.make_image(fill_color="black", back_color="white")

# Resize for SD
w, h = qr_img.size
w = (w + 255) // 256 * 256
h = (h + 255) // 256 * 256

bg = Image.new("L", (w, h), 128)

coords = ((w - qr_img.size[0]) // 2, (h - qr_img.size[1]) // 2)
bg.paste(qr_img.convert("L"), coords)

# -----------------------------
# Generate Images
# -----------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

for i in range(NUM_IMAGES):

    print(f"Generating image {i+1}/{NUM_IMAGES}")

    generator = torch.Generator(DEVICE).manual_seed(torch.randint(0,999999,(1,)).item())

    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=bg,
        guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=CONTROL_SCALE,
        num_inference_steps=STEPS,
        generator=generator
    ).images[0]

    path = f"{OUTPUT_DIR}/qr_art_{i}.png"
    image.save(path)

    print("Saved:", path)

print("Done.")

In [ ]:
"""
Patri AI-Based Cooking Oil — Branded Scannable QR Code Generator
=================================================================
Generates high-quality, SCANNABLE artistic QR codes with the Patri brand aesthetic.

Key fixes over the original:
  1. controlnet_conditioning_scale lowered to 0.85–1.1  (keeps QR readable)
  2. guidance_scale raised to 9–12                       (strong prompt adherence)
  3. ERROR_CORRECT_H used                                (30% data recovery headroom)
  4. Auto-scan verification via pyzbar before saving
  5. Retry loop — keeps generating until a scannable result is found
  6. Optional: paste the Patri logo watermark in the centre quiet zone

Usage
-----
pip install torch diffusers transformers accelerate xformers
pip install qrcode[pil] pillow pyzbar

python patri_qr_generator.py
"""

import os, io, random, sys
import torch
import qrcode
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance

# ── Optional: pyzbar for scan-verification ──────────────────────────────────
try:
    from pyzbar.pyzbar import decode as zbar_decode
    HAS_ZBAR = True
except ImportError:
    HAS_ZBAR = False
    print("⚠  pyzbar not found — scan-verification disabled (install: pip install pyzbar)")

# ── Optional: diffusers for AI-styled QR ────────────────────────────────────
try:
    from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, DDIMScheduler
    HAS_DIFFUSERS = True
except ImportError:
    HAS_DIFFUSERS = False
    print("⚠  diffusers not found — falling back to styled non-AI QR")

# =============================================================================
# CONFIGURATION
# =============================================================================
QR_CONTENT   = "https://huggingface.co/spaces/kittu125/FamilyBasedOilBlend"
OUTPUT_DIR   = "./patri_output"
MODEL_CACHE  = "./models"
LOGO_PATH    = "C:/Users/pkitt/Downloads/Patri Image.png"         # place your logo file here (optional)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

# ── QR settings ─────────────────────────────────────────────────────────────
QR_SIZE      = 768   # pixels — multiple of 64 for SD
QR_BOX       = 10
QR_BORDER    = 4

# ── AI generation settings ──────────────────────────────────────────────────
# THESE VALUES ARE THE KEY FIX:
#   controlnet_scale  ≤ 1.1  → keeps QR structure intact (scannable)
#   guidance_scale    ≥ 9    → strong aesthetic prompt
CONTROLNET_SCALE = 0.95    # was 1.8 in original — that's why QRs were unreadable
GUIDANCE_SCALE   = 10.0
STEPS            = 40
MAX_RETRIES      = 8       # attempts per style before giving up

# ── Brand prompts ────────────────────────────────────────────────────────────
PROMPTS = [
    # Seed / nature style (matches your existing outputs)
    (
        "top-down macro photograph of sesame seeds and green mung beans arranged on black lacquer, "
        "premium cooking oil brand, crisp studio lighting, olive green and ivory palette, "
        "scattered seeds forming organic patterns, photorealistic, 8k detail",
        "blurry, text, watermark, human, face, hands, bottle, distorted, low quality"
    ),
    # Oil drop / leaf style
    (
        "golden cooking oil droplets splashing on fresh green basil leaves, dark background, "
        "macro photography, glistening oil texture, vibrant green and gold, premium food brand, "
        "bokeh, studio lighting, photorealistic",
        "blurry, watermark, face, distorted pattern, low contrast, washed out"
    ),
    # Geometric / Indian motif style
    (
        "intricate Indian rangoli pattern with mustard seeds and green lentils on dark surface, "
        "overhead flat lay, earthy gold and forest green palette, premium cooking oil brand Patri, "
        "symmetrical mandala, photorealistic macro detail",
        "blurry, plastic, synthetic, text, face, hands, watermark"
    ),
    # Minimalist grain style
    (
        "flat lay of sunflower seeds and green pistachios on obsidian black marble, "
        "premium cooking oil advertisement, elegant minimal composition, "
        "soft studio lighting, gold and sage green tones, ultra sharp 8k",
        "blurry, crowded, messy, text, face, watermark, noisy"
    ),
]

# =============================================================================
# HELPERS
# =============================================================================

def make_qr_image(content: str, size: int = QR_SIZE) -> Image.Image:
    """Generate a high-error-correction QR code at the target pixel size."""
    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_H,  # 30% restoration
        box_size=QR_BOX,
        border=QR_BORDER,
    )
    qr.add_data(content)
    qr.make(fit=True)
    img = qr.make_image(fill_color="black", back_color="white").convert("RGB")

    # Pad to square multiple-of-64 for Stable Diffusion
    target = (size, size)
    canvas = Image.new("RGB", target, (255, 255, 255))
    img.thumbnail((size - 32, size - 32), Image.LANCZOS)
    offset = ((size - img.width) // 2, (size - img.height) // 2)
    canvas.paste(img, offset)
    return canvas


def is_scannable(img: Image.Image, expected_url: str) -> bool:
    """Return True if pyzbar can read the expected URL from the image."""
    if not HAS_ZBAR:
        return True   # assume OK if library missing
    results = zbar_decode(img)
    for r in results:
        if r.data.decode("utf-8").strip() == expected_url.strip():
            return True
    return False


def add_logo_to_qr(qr_img: Image.Image, logo_path: str) -> Image.Image:
    """Paste the Patri logo into the centre quiet zone (≤ 30% of QR area)."""
    if not os.path.exists(logo_path):
        return qr_img
    logo = Image.open(logo_path).convert("RGBA")
    # Logo should be at most 28% of QR dimension
    max_logo = int(qr_img.width * 0.28)
    logo.thumbnail((max_logo, max_logo), Image.LANCZOS)
    x = (qr_img.width  - logo.width)  // 2
    y = (qr_img.height - logo.height) // 2
    # White backing so logo doesn't interfere with finder patterns
    backing = Image.new("RGBA", logo.size, (255, 255, 255, 255))
    qr_rgba = qr_img.convert("RGBA")
    qr_rgba.paste(backing, (x, y))
    qr_rgba.paste(logo, (x, y), logo)
    return qr_rgba.convert("RGB")


def styled_qr_no_ai(qr_img: Image.Image) -> Image.Image:
    """
    Fallback: apply Patri brand colours without AI.
    Tints black modules dark-green, white modules ivory/gold.
    Remains scannable.
    """
    img = qr_img.convert("RGB")
    pixels = img.load()
    w, h = img.size
    for y in range(h):
        for x in range(w):
            r, g, b = pixels[x, y]
            brightness = (r + g + b) / 3
            if brightness < 128:
                # dark module → deep forest green
                pixels[x, y] = (30, 80, 30)
            else:
                # light module → warm ivory
                pixels[x, y] = (245, 235, 200)
    # subtle vignette
    vignette = Image.new("L", img.size, 0)
    draw = ImageDraw.Draw(vignette)
    for i in range(min(w, h) // 6):
        alpha = int(30 * i / (min(w, h) // 6))
        draw.rectangle([i, i, w - i, h - i], outline=alpha)
    img = Image.composite(
        img,
        Image.new("RGB", img.size, (20, 60, 20)),
        ImageEnhance.Brightness(vignette).enhance(0.5)
    )
    return img


# =============================================================================
# AI PIPELINE
# =============================================================================

def load_pipeline():
    print("📦 Loading ControlNet QR model …")
    controlnet = ControlNetModel.from_pretrained(
        "monster-labs/control_v1p_sd15_qrcode_monster",
        cache_dir=MODEL_CACHE,
        torch_dtype=DTYPE,
    )
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        cache_dir=MODEL_CACHE,
        safety_checker=None,
        torch_dtype=DTYPE,
    ).to(DEVICE)

    # DDIM scheduler — more stable than PNDM for ControlNet QR
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

    if DEVICE == "cuda":
        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception:
            pass
    return pipe


def generate_ai_qr(pipe, qr_control: Image.Image, prompt: str, neg: str, seed: int) -> Image.Image:
    generator = torch.Generator(DEVICE).manual_seed(seed)
    result = pipe(
        prompt=prompt,
        negative_prompt=neg,
        image=qr_control,
        guidance_scale=GUIDANCE_SCALE,
        controlnet_conditioning_scale=CONTROLNET_SCALE,  # ← THE KEY FIX
        num_inference_steps=STEPS,
        generator=generator,
        width=QR_SIZE,
        height=QR_SIZE,
    ).images[0]
    return result


# =============================================================================
# MAIN
# =============================================================================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Base QR control image
    print(f"🔲 Generating QR for: {QR_CONTENT}")
    qr_control = make_qr_image(QR_CONTENT, QR_SIZE)
    qr_control.save(os.path.join(OUTPUT_DIR, "qr_control_base.png"))
    print("   Saved control image: qr_control_base.png")

    # 2. Verify the base QR itself is scannable
    if HAS_ZBAR:
        assert is_scannable(qr_control, QR_CONTENT), "❌ Base QR is not scannable — check URL!"
        print("   ✅ Base QR verified scannable")

    # 3. Styled non-AI fallback (always generated — guaranteed scannable)
    styled_fallback = styled_qr_no_ai(qr_control)
    styled_with_logo = add_logo_to_qr(styled_fallback, LOGO_PATH)
    fallback_path = os.path.join(OUTPUT_DIR, "patri_styled_guaranteed.png")
    styled_with_logo.save(fallback_path)
    print(f"\n🎨 Saved guaranteed-scannable styled QR → {fallback_path}")

    # 4. AI-enhanced versions (if diffusers available)
    if not HAS_DIFFUSERS:
        print("\n✅ Done. Only styled QR generated (install diffusers for AI art version).")
        return

    pipe = load_pipeline()
    saved = 0

    for style_idx, (prompt, neg) in enumerate(PROMPTS):
        print(f"\n🎨 Style {style_idx + 1}/{len(PROMPTS)}: {prompt[:60]}…")
        for attempt in range(MAX_RETRIES):
            seed = random.randint(0, 999_999)
            print(f"   Attempt {attempt + 1}/{MAX_RETRIES}  seed={seed} ", end="", flush=True)
            try:
                result = generate_ai_qr(pipe, qr_control, prompt, neg, seed)
            except Exception as e:
                print(f"ERROR: {e}")
                continue

            scannable = is_scannable(result, QR_CONTENT)
            print("✅ SCANNABLE" if scannable else "❌ not scannable")

            if scannable:
                # Optionally add logo to centre
                result = add_logo_to_qr(result, LOGO_PATH)
                out_path = os.path.join(OUTPUT_DIR, f"patri_ai_style{style_idx+1}_seed{seed}.png")
                result.save(out_path)
                print(f"   💾 Saved → {out_path}")
                saved += 1
                break  # move to next style
        else:
            print(f"   ⚠  No scannable result found for style {style_idx+1} after {MAX_RETRIES} attempts.")
            print(f"      Tip: Try lowering CONTROLNET_SCALE further (current: {CONTROLNET_SCALE})")

    print(f"\n✅ Done. {saved}/{len(PROMPTS)} AI QR codes saved to ./{OUTPUT_DIR}/")
    print(f"   + 1 guaranteed-scannable styled QR always saved.")


if __name__ == "__main__":
    main()

⚠  pyzbar not found — scan-verification disabled (install: pip install pyzbar)
🔲 Generating QR for: https://huggingface.co/spaces/kittu125/FamilyBasedOilBlend
   Saved control image: qr_control_base.png

🎨 Saved guaranteed-scannable styled QR → ./patri_output\patri_styled_guaranteed.png
📦 Loading ControlNet QR model …


C:\Users\pkitt\anaconda3\Lib\site-packages\huggingface_hub\utils\_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: models\models--runwayml--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information


🎨 Style 1/4: top-down macro photograph of sesame seeds and green mung bea…
   Attempt 1/8  seed=322249 

  0%|          | 0/40 [00:00<?, ?it/s]